# Aggregate latest forecasts

Fetches the latest forecast from each source and compares them over the UK, using the **Met Office** UK 2 km model as the benchmark:

| Source | Model | Steps | Loader |
|---|---|---|---|
| **Met Office** (benchmark) | UK 2 km, from the Weather DataHub | hourly | `MetOfficeLoader` in [`loaders/met_office.py`](../loaders/met_office.py) |
| **ECMWF IFS** | global 0.25° physics-based, from ECMWF open data | 3-hourly | `ECMWFLoader` in [`loaders/ecmwf.py`](../loaders/ecmwf.py) |
| **ECMWF AIFS** | global 0.25° machine-learning (AIFS Single), same open data | 6-hourly | `ECMWFLoader` |
| **Met Office global** | global 10 km, from the Met Office's public archive on AWS | hourly | `MetOfficeGlobalLoader` in [`loaders/met_office.py`](../loaders/met_office.py) |
| **NOAA GFS** | global 0.25° physics-based, from NOAA's AWS bucket | hourly | `GFSLoader` in [`loaders/gfs.py`](../loaders/gfs.py) |
| **NOAA AIGFS** | global 0.25° AI model, from NOAA's AWS bucket | 6-hourly | `GFSLoader` |
| **DWD ICON-EU** | European ~7 km, from DWD open data | hourly | `ICONEULoader` in [`loaders/dwd.py`](../loaders/dwd.py) |
| **DWD ICON-D2** | ~2 km for Germany and neighbours, same open data: **regional**, covering London and eastern England only (to 48 h) | hourly | `ICOND2Loader` in [`loaders/dwd.py`](../loaders/dwd.py) |
| **Canada GDPS** | global 15 km (GEM), from the MSC Datamart | hourly | `GDPSLoader` in [`loaders/msc.py`](../loaders/msc.py) |

The Met Office DataHub needs `MET_OFFICE_API_KEY` in the project-root `.env`, and its downloads count towards the data allowance. The other sources need no key: Met Office archive (CC BY-SA 4.0), ECMWF (CC BY 4.0, data © ECMWF), NOAA (public domain), DWD (CC BY 4.0, data © Deutscher Wetterdienst) and Environment Canada (Open Government Licence – Canada). The Met Office global model is the largest download, at roughly 900 MB per new 72-hour run (kept only while it's cropped, then deleted), since its archive only has whole global fields.

Every loader returns the same standard format (see [`loaders/standard.py`](../loaders/standard.py)), so this notebook doesn't need to know where data came from. New sources are added in `loaders.forecast_sources` and checked by the tests in [`tests/`](../tests/). Every other source is shown out to `HORIZON_HOURS` (72 h) after the benchmark's run time, beyond the benchmark's own forecast (~22 h with the current DataHub order); the scores only use the hours the benchmark covers. Rainfall is compared as 6-hour totals, since AIFS steps are 6-hourly. Forecasts are kept in a store under `data/standard/` next to this notebook (see [`loaders/store.py`](../loaders/store.py)): only the latest set, cropped to the UK box and compressed (~30 MB). A source is only downloaded when it has a new run, its raw files are deleted once it's saved, and older runs are deleted after each refresh.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from functools import reduce
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import xarray as xr
from plotly.subplots import make_subplots

# Interactive figures: shown by VS Code/JupyterLab, and loading plotly.js from its CDN elsewhere
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Make the project-root loaders package importable from this folder
sys.path.insert(0, str(Path.cwd().parent))
from loaders import BENCHMARK
from loaders.standard import period_totals, regrid_to_box, run_time, wind_speed
from loaders.store import UK_EXTENT, ForecastStore

# UK_EXTENT (-13, 4, 49.5, 61) is the box sources are cropped to and the maps show (lon_min, lon_max, lat_min, lat_max)
store = ForecastStore(data_dir="data", extent=UK_EXTENT)
# Mainland UK, which lies inside every source's grid: used for the scores
SCORE_EXTENT = (-8, 2, 50, 59)
# How far ahead to show forecasts, from the benchmark's run time (hours)
HORIZON_HOURS = 72
# Rainfall is compared as totals over this many hours, ending at each valid time
RAIN_HOURS = 6
# Sources to show in the maps (None for all); everything else always uses every source
MAP_SOURCES = None

## Latest forecasts: Met Office benchmark and every other source

In [ ]:
# Latest run of the benchmark (the first active DataHub order), and of every other source out to
# HORIZON_HOURS after the benchmark's run time. Only runs not already stored are downloaded; older runs
# are then deleted
forecasts = store.refresh(horizon_hours=HORIZON_HOURS)
benchmark = forecasts[BENCHMARK]
bm_end = pd.Timestamp(benchmark.valid_time.values[-1])

for name, ds in forecasts.items():
    print(f"{name}: {run_time(ds):%Y-%m-%d %H:%M} UTC run, valid to {pd.Timestamp(ds.valid_time.values[-1]):%Y-%m-%d %H:%M} UTC, "
          f"{ds.attrs['step_hours']}-hourly at first")
for name, error in store.failures.items():
    print(f"{name} FAILED ({'using its previous run' if name in forecasts else 'left out'}): {error}")
print(f"Data folder: {store.size_mb():.0f} MB")

## Fields to compare

Maps use the valid times shared by every source except the benchmark, so they span the whole horizon; the benchmark's panels are blank after its forecast ends.

In [ ]:
rain = {name: period_totals(ds, RAIN_HOURS) for name, ds in forecasts.items()}
wind = {name: wind_speed(ds) for name, ds in forecasts.items()}
runs = {name: run_time(ds) for name, ds in forecasts.items()}


def shared_times(fields):
    return reduce(np.intersect1d, [da.valid_time.values for da in fields.values()])


def others(fields):
    return {name: da for name, da in fields.items() if name != BENCHMARK}


rain_times, wind_times = shared_times(others(rain)), shared_times(others(wind))
for label, times in [("rain", rain_times), ("wind", wind_times)]:
    if len(times):
        print(f"Shared {label} times: {len(times)}, {pd.Timestamp(times[0]):%a %d %H:%M} to {pd.Timestamp(times[-1]):%a %d %H:%M} UTC")
    else:
        print(f"No {label} times shared by every source (see any failures above), so no {label} maps")

## Maps of each source, benchmark first

Set `MAP_SOURCES` above to map only some sources.

In [ ]:
def compare_maps(fields, times, title, cmap, vmax, units, max_cols=4):
    """Grid of maps of each source at each valid time, source by source.

    Each row holds one source's times, or several sources side by side when there are few times, so
    the grid is up to max_cols maps wide.
    """
    panels = [(name, t) for name in fields for t in times]
    ncols = len(times) * max(1, max_cols // len(times))
    nrows = -(-len(panels) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.9 * nrows), squeeze=False,
                             subplot_kw={"projection": ccrs.PlateCarree()}, layout="constrained")
    for ax, (name, t) in zip(axes.flat, panels):
        ax.set_extent(UK_EXTENT, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.set_title(f"{name} ({runs[name]:%d %H}Z run)\n{pd.Timestamp(t):%a %d %H:%M} UTC", fontsize=9)
        if t not in fields[name].valid_time:  # e.g. beyond the benchmark's forecast
            ax.text(0.5, 0.5, "Not forecast", transform=ax.transAxes, ha="center", va="center", color="dimgrey",
                    bbox={"facecolor": "white", "edgecolor": "lightgrey", "boxstyle": "round"})
            continue
        field = fields[name].sel(valid_time=t)
        mesh = ax.pcolormesh(field.longitude, field.latitude, field, transform=ccrs.PlateCarree(),
                             cmap=cmap, vmin=0, vmax=vmax, shading="auto")
    for ax in axes.flat[len(panels):]:
        ax.set_visible(False)
    fig.colorbar(mesh, ax=axes, shrink=0.6, label=units)
    fig.suptitle(title)
    plt.show()


# Show up to 4 shared times, spread across the forecast
def pick(times, n=4):
    return times[np.linspace(0, len(times) - 1, min(n, len(times))).round().astype(int)]


def shown(fields):
    return {name: da for name, da in fields.items() if MAP_SOURCES is None or name in MAP_SOURCES}


if len(rain_times):
    compare_maps(shown(rain), pick(rain_times), f"{RAIN_HOURS}-hour rainfall", "Blues", vmax=15, units="mm")
if len(wind_times):
    compare_maps(shown(wind), pick(wind_times), "10 m wind speed", "viridis", vmax=20, units="m s$^{-1}$")

## Time series at a point

Each source is plotted at its own step spacing (hourly, except IFS 3-hourly and AIFS and AIGFS 6-hourly). Rainfall is the total over the previous 6 hours in every case. The dashed line marks the end of the benchmark's forecast.

The chart is interactive: click a source in the legend to hide or show it (in both panels), double-click one to show only that source (double-click again for all), hover to compare values, and drag to zoom (double-click the chart to reset).

In [ ]:
LAT, LON = 51.51, -0.13  # London


def at_point(da):
    if da.latitude.ndim == 1:
        return da.sel(latitude=LAT, longitude=LON, method="nearest")
    # Met Office grid has 2D lat/lon, so take the closest grid cell
    dist = (da.latitude - LAT) ** 2 + ((da.longitude - LON) * np.cos(np.radians(LAT))) ** 2
    y, x = np.unravel_index(np.argmin(dist.values), dist.shape)
    return da.isel(y=y, x=x)


fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=(f"{RAIN_HOURS}-hour rainfall, London", "10 m wind speed, London"))
colours = pio.templates["plotly"].layout.colorway
for i, name in enumerate(forecasts):
    style = {"name": f"{name} ({runs[name]:%d %H}Z run)", "legendgroup": name, "mode": "lines+markers",
             "line": {"color": colours[i % len(colours)], "width": 3 if name == BENCHMARK else 1.5},
             "marker": {"size": 4}}
    for row, (field, units) in enumerate([(rain, "mm"), (wind, "m/s")], start=1):
        series = at_point(field[name])
        fig.add_trace(go.Scatter(x=series.valid_time.values, y=series.values, showlegend=row == 1,
                                 hovertemplate=f"%{{y:.2f}} {units}", **style), row=row, col=1)

# Where the benchmark's forecast ends (a shape, as add_vline's label doesn't support date axes)
for row in (1, 2):
    fig.add_shape(type="line", x0=bm_end, x1=bm_end, y0=0, y1=1, yref=f"y{'' if row == 1 else row} domain",
                  line={"color": "grey", "dash": "dash", "width": 1}, row=row, col=1)
fig.add_annotation(x=bm_end, y=1, yref="y2 domain", text="benchmark ends", showarrow=False,
                   xanchor="left", yanchor="top", font={"size": 10, "color": "grey"})
fig.update_yaxes(title_text="mm", rangemode="tozero", row=1, col=1)
fig.update_yaxes(title_text="m s⁻¹", row=2, col=1)
fig.update_xaxes(title_text="Valid time (UTC)", row=2, col=1)
fig.update_layout(height=700, hovermode="x unified", template="plotly_white",
                  legend={"title": "Click to hide/show, double-click to isolate"}, margin={"t": 60})
fig.show()

## Scores against the benchmark

Each source and the benchmark are averaged onto a common 0.5° grid over mainland UK, and compared at every valid time they share (not only those shared by all sources), so only within the benchmark's forecast. Bias is the source's mean minus the benchmark's, so positive means wetter or windier. The benchmark is a forecast too, not observations, so these measure agreement with the Met Office, not accuracy.

In [ ]:
def scores(field):
    """Agreement of each source's field with the benchmark's, on the common grid at shared valid times."""
    bm = regrid_to_box(field[BENCHMARK], SCORE_EXTENT)
    rows = {}
    for name in forecasts:
        if name == BENCHMARK:
            continue
        a, b = xr.align(bm, regrid_to_box(field[name], SCORE_EXTENT), join="inner")
        ok = np.isfinite(a.values) & np.isfinite(b.values)
        diff = b.values[ok] - a.values[ok]
        rows[name] = {"shared times": a.valid_time.size, "bias": diff.mean(), "RMSE": np.sqrt((diff ** 2).mean()),
                      "correlation": np.corrcoef(a.values[ok], b.values[ok])[0, 1]}
    return pd.DataFrame(rows).T.astype({"shared times": int})


pd.concat({f"{RAIN_HOURS}-hour rainfall (mm)": scores(rain), "10 m wind speed (m/s)": scores(wind)}, axis=1).round(2)